<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 컴퓨터 비전 · 04. U-Net

## U-Net: Convolutional Networks for Biomedical Image Segmentation

- **원 논문:** [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
- **저자 / 발표:** Olaf Ronneberger, Philipp Fischer, Thomas Brox · MICCAI 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch tensor, 합성곱, 역전파의 기초
- **로컬 학습 데이터:** `data/field_curriculum/vision_shapes.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** contracting/expanding path와 skip concatenation으로 mask를 분할한다.

**축소하거나 생략한 부분:** 572×572 biomedical images, valid convolution, cropping, elastic deformation 대신 16×16 합성 mask와 same-padding 1-level U-Net을 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2 Fig. 1: contracting/expansive path | Task 1 MiniUNet | 입출력 해상도 |
| §2: cropped feature-map concatenation | Task 1 skip concat | decoder channels |
| §2.1 Eq. (1): pixel-wise softmax/CE | Task 2 BCE 이진 대응 | loss 감소 |
| §3 Fig. 3: overlap quality | Task 3 Dice | mask plot |

## 핵심 아이디어와 수식

$$\operatorname{Dice}(P,Y)=\frac{2\sum_i P_iY_i+\epsilon}{\sum_iP_i+\sum_iY_i+\epsilon}$$

**기호 해설:** P는 예측 mask, Y는 정답 mask, i는 pixel, ε은 0 나눗셈 방지 상수다.

**코드 대응:** Task 1은 encoder/decoder와 skip을 만들고 Task 2가 BCE, Task 3이 Dice를 구현한다.

**입력과 출력 shape:** 입력 [B,1,16,16] → bottleneck [B,12,8,8] → logits [B,1,16,16].

**포트폴리오 구현 범위:** 논문의 4-level 깊이와 crop은 줄였지만 localization skip과 pixel loss는 보존한다.

아래 코드는 결과만 내는 라이브러리 호출로 핵심을 감추지 않는다. config, 논문 고유
연산, 모델의 `forward`, 목적함수, `train_step`, 평가 함수를 순서대로 직접 작성한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,np.load,torch.from_numpy,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.from_numpy(ndarray)</code></summary>

#### `torch.from_numpy(ndarray)`

- **역할:** NumPy 배열과 메모리를 공유하는 CPU 텐서를 만듭니다.
- **입력·반환:** NumPy `ndarray`를 받고 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 복사 없이 준비된 NumPy 데이터를 모델 입력으로 바꿉니다.
- **주의점:** 한쪽의 in-place 변경이 다른 쪽에도 반영될 수 있습니다.
- **공식 문서:** [torch.from_numpy](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import itertools
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/vision_shapes.npz")
raw = np.load(dataset_path)
images = torch.from_numpy(raw["images"]).float()
labels = torch.from_numpy(raw["labels"]).long()
masks = torch.from_numpy(raw["masks"]).float()
boxes = torch.from_numpy(raw["boxes"]).float()
train_idx = torch.from_numpy(raw["train_idx"][:128]).long()
test_idx = torch.from_numpy(raw["test_idx"][:64]).long()
images, labels, masks, boxes = ACCELERATOR.move(
    images,
    labels,
    masks,
    boxes,
)
train_idx, test_idx = ACCELERATOR.move(train_idx, test_idx)
class_names = raw["class_names"].tolist()
x_train = images[train_idx]
y_train = labels[train_idx]
assert images.shape == (480, 1, 16, 16)
assert masks.shape == (480, 16, 16)
assert boxes.shape == (480, 4)
print(ACCELERATOR.summary())
print(dataset_path.name, images.shape, class_names)

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Fig. 1: contracting/expansive path
- **노트북에서 구현할 부분:** Task 1 MiniUNet
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** 입출력 해상도

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,nn.Sequential,nn.Conv2d,nn.ReLU,nn.ConvTranspose2d,F.max_pool2d,torch.cat -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Conv2d(in_channels, out_channels, kernel_size, ...)</code></summary>

#### `nn.Conv2d(in_channels, out_channels, kernel_size, ...)`

- **역할:** 2차원 convolution 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_{n,o,h,w}=b_o+\sum_{c,i,j}W_{o,c,i,j}x_{n,c,h\cdot s_h+i-p_h,w\cdot s_w+j-p_w}
$$

- **기호:** $n,o,c$는 batch·출력·입력 channel, $i,j$는 kernel 위치입니다.
- **수식 → 코드:** `kernel_size`, `stride`, `padding`이 수식의 합 범위와 입력 좌표를 정합니다.
- **직관:** 같은 작은 filter를 모든 공간 위치에 공유해 지역 패턴을 찾습니다.
- **shape 확인:** `[N,C_in,H,W] -> [N,C_out,H_out,W_out]`; 공간 크기는 stride·padding에 따라 달라집니다.
- **공식 문서:** [nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>nn.ConvTranspose2d(in_channels, out_channels, kernel_size, ...)</code></summary>

#### `nn.ConvTranspose2d(in_channels, out_channels, kernel_size, ...)`

- **역할:** 학습 가능한 2차원 전치 convolution을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
H_{out}=(H_{in}-1)s-2p+d(k-1)+o+1
$$

- **기호:** $s,p,d,k,o$는 stride, padding, dilation, kernel, output padding입니다.
- **수식 → 코드:** 입력 위치의 값을 kernel로 펼쳐 겹치는 출력 위치에 더합니다.
- **직관:** convolution의 선형 연산을 전치한 것으로 단순한 역함수는 아닙니다.
- **shape 확인:** `[N,C_in,H,W] -> [N,C_out,H_out,W_out]`입니다.
- **공식 문서:** [nn.ConvTranspose2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)

</details>

<details>
<summary><code>F.max_pool2d(input, kernel_size, stride=None, padding=0, ...)</code></summary>

#### `F.max_pool2d(input, kernel_size, stride=None, padding=0, ...)`

- **역할:** 각 공간 window의 최댓값으로 feature map을 축소합니다.
- **입력·반환:** 4D 텐서와 window 설정을 받고 축소된 feature map을 반환합니다.
- **이 셀에서 쓰는 이유:** 강한 지역 반응을 보존하면서 공간 해상도와 계산량을 줄입니다.
- **주의점:** stride·padding·ceil_mode가 출력 크기와 경계 window를 바꿉니다.

##### 관련 수식과 코드 연결

$$
H_{out}=\left\lfloor\frac{H_{in}+2p-d(k-1)-1}{s}+1\right\rfloor
$$

- **기호:** $k,s,p,d$는 kernel, stride, padding, dilation입니다.
- **수식 → 코드:** 각 window에서 max 또는 평균을 계산하고 stride만큼 이동합니다.
- **직관:** channel 수는 유지하면서 공간 정보를 지역 대표값으로 압축합니다.
- **shape 확인:** `[N,C,H,W] -> [N,C,H_out,W_out]`입니다.
- **공식 문서:** [F.max_pool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.max_pool2d.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass
class UNetConfig:
    encoder_channels: int = 6
    bottleneck_channels: int = 12
    learning_rate: float = 0.02
    steps: int = 16


def make_double_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.ReLU(),
    )


class MiniUNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        c1 = config.encoder_channels
        c2 = config.bottleneck_channels
        self.encoder = make_double_conv(1, c1)
        self.bottleneck = make_double_conv(c1, c2)
        self.upsample = nn.ConvTranspose2d(c2, c1, 2, stride=2)
        self.decoder = make_double_conv(c1 * 2, c1)
        self.mask_head = nn.Conv2d(c1, 1, kernel_size=1)

    def encode(self, inputs):
        skip = self.encoder(inputs)
        low = self.bottleneck(F.max_pool2d(skip, 2))
        return skip, low

    def forward(self, inputs):
        skip, low = self.encode(inputs)
        upsampled = self.upsample(low)
        decoded = self.decoder(torch.cat((upsampled, skip), dim=1))
        return self.mask_head(decoded)


config = UNetConfig()
model = MiniUNet(config).to(DEVICE)
assert model(images[:2]).shape == (2, 1, 16, 16)
assert model.decoder[0].in_channels == 12

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2: cropped feature-map concatenation
- **노트북에서 구현할 부분:** Task 1 skip concat
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** decoder channels

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.binary_cross_entropy_with_logits,Module.train,Optimizer.zero_grad,Tensor.backward,Optimizer.step,Tensor.detach,torch.optim.Adam,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.binary_cross_entropy_with_logits(input, target, ...)</code></summary>

#### `F.binary_cross_entropy_with_logits(input, target, ...)`

- **역할:** sigmoid와 binary cross entropy를 안정적으로 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{BCE}=-\frac{1}{N}\sum_i\left[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))\right]
$$

- **기호:** $z_i$는 logit, $y_i\in[0,1]$은 target, $\sigma$는 sigmoid입니다.
- **수식 → 코드:** sigmoid와 BCE를 한 함수에서 log-sum-exp 형태로 안정적으로 계산합니다.
- **직관:** 각 항을 독립적인 이진 선택으로 보고 정답 쪽 확률의 음의 로그를 최소화합니다.
- **shape 확인:** logit과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.binary_cross_entropy_with_logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def segmentation_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)


def train_segmentation_step(model, optimizer, inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = segmentation_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


target_masks = masks[train_idx, None]
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    loss = train_segmentation_step(
        model,
        optimizer,
        x_train,
        target_masks,
    )
    losses.append(loss)
assert min(losses[3:]) < losses[0]

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3 Fig. 3: overlap quality
- **노트북에서 구현할 부분:** Task 3 Dice
- **구현 이유:** 논문의 계산 경로를 작은 tensor로 보존하면서 각 중간값을
  assertion으로 관찰하기 위해서다.
- **완료 증거:** mask plot

먼저 입력 shape와 기대 출력 shape를 종이에 적은 뒤 아래 TODO를 구현한다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.sum,Module.eval,torch.no_grad -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def dice_score(predictions, targets, epsilon=1.0):
    dimensions = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dimensions)
    total = predictions.sum(dim=dimensions) + targets.sum(dim=dimensions)
    return ((2 * intersection + epsilon) / (total + epsilon)).mean()


def evaluate_segmentation(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        probabilities = model(inputs).sigmoid()
        predictions = (probabilities > 0.5).float()
    return float(dice_score(predictions, targets)), predictions


truth = masks[test_idx, None]
dice, predictions = evaluate_segmentation(model, images[test_idx], truth)
assert 0.0 <= dice <= 1.0
figure, axes = plt.subplots(1, 3, figsize=(8, 2.7))
axes[0].plot(losses)
axes[0].set_title("pixel BCE")
axes[1].imshow(truth[0, 0].detach().cpu())
axes[1].set_title("target")
axes[2].imshow(predictions[0, 0].detach().cpu())
axes[2].set_title(f"prediction Dice={dice:.2f}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 572×572 biomedical images, valid convolution, cropping, elastic deformation 대신 16×16 합성 mask와 same-padding 1-level U-Net을 사용한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.